# Stepwise Visualization Notebook (Normal Case)

This notebook visualizes **step1 ~ step5** outputs from the quantitative pipeline.
Current notebook explanation follows the **previous-version metric definition** (Step4/Step5 independent of direct Step3 input).

In [ ]:
from pathlib import Path

# ---- Configure case here ----
CASE = "Normal_1"  # e.g. Normal_1, Normal_2

REPO = Path.cwd()
if not (REPO / "scripts").exists() and (REPO.parent / "scripts").exists():
    REPO = REPO.parent

CT_PATH = REPO / f"ASOCA2020/Normal/CTCA_nii/{CASE}.nii.gz"
GT_MASK_PATH = REPO / f"ASOCA2020/Normal/Annotations_nii/{CASE}.nii.gz"
GT_CL_PATH = REPO / f"ASOCA2020/Normal/Centerlines/{CASE}.vtp"
OUT_DIR = REPO / f"outputs/quant/{CASE}"

print("repo:", REPO)
print("case:", CASE)
print("out:", OUT_DIR)


In [ ]:
import json
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import vtk  # type: ignore
except Exception as exc:
    raise RuntimeError("vtk is required for centerline visualization") from exc


def must_exist(path: Path):
    if not path.exists():
        raise FileNotFoundError(path)


def load_json(path: Path):
    must_exist(path)
    return json.loads(path.read_text(encoding="utf-8"))


def load_nifti(path: Path):
    must_exist(path)
    img = nib.load(str(path))
    arr = np.asarray(img.dataobj)
    return arr, img.affine, img.header.get_zooms()[:3]


def voxel_indices_to_world(ijk: np.ndarray, affine: np.ndarray) -> np.ndarray:
    ijk = np.asarray(ijk, dtype=float)
    if ijk.size == 0:
        return np.zeros((0, 3), dtype=float)
    if ijk.ndim == 1:
        ijk = ijk[None, :]
    hom = np.hstack([ijk, np.ones((ijk.shape[0], 1), dtype=float)])
    return (affine @ hom.T).T[:, :3]


def repair_alignment_shift_from_report(report_path: Path):
    if not report_path.exists():
        return None
    report = load_json(report_path)
    shift = report.get("alignment", {}).get("best_shift_world")
    if shift is None:
        return None
    # New repair output already applies the shift into repaired.vtp.
    if report.get("alignment_applied_to_output", False):
        return None
    return np.asarray(shift, dtype=float)


def mask_center(mask: np.ndarray):
    coords = np.argwhere(mask > 0)
    if coords.size == 0:
        return tuple((np.array(mask.shape) // 2).tolist())
    return tuple(np.median(coords, axis=0).astype(int).tolist())


def show_seg_overlay(ct, gt, pred, title="Step1 segmentation overlay", metrics=None):
    from matplotlib.lines import Line2D

    c = mask_center(gt)
    z, y, x = c
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # axial
    axes[0].imshow(ct[:, :, x].T, cmap="gray", origin="lower")
    axes[0].contour(gt[:, :, x].T, levels=[0.5], colors=["lime"], linewidths=1.0)
    axes[0].contour(pred[:, :, x].T, levels=[0.5], colors=["red"], linewidths=1.0)
    axes[0].set_title(f"Axial @ x={x}")

    # coronal
    axes[1].imshow(ct[:, y, :].T, cmap="gray", origin="lower")
    axes[1].contour(gt[:, y, :].T, levels=[0.5], colors=["lime"], linewidths=1.0)
    axes[1].contour(pred[:, y, :].T, levels=[0.5], colors=["red"], linewidths=1.0)
    axes[1].set_title(f"Coronal @ y={y}")

    # sagittal
    axes[2].imshow(ct[z, :, :].T, cmap="gray", origin="lower")
    axes[2].contour(gt[z, :, :].T, levels=[0.5], colors=["lime"], linewidths=1.0)
    axes[2].contour(pred[z, :, :].T, levels=[0.5], colors=["red"], linewidths=1.0)
    axes[2].set_title(f"Sagittal @ z={z}")

    for ax in axes:
        ax.set_axis_off()

    legend_handles = [
        Line2D([0], [0], color="gray", lw=4, label="CT slice"),
        Line2D([0], [0], color="lime", lw=2, label="GT contour"),
        Line2D([0], [0], color="red", lw=2, label="Pred contour"),
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=3, frameon=True, bbox_to_anchor=(0.5, -0.02))

    if metrics is not None:
        key_map = [
            ("dice", "Dice"),
            ("hd95_mm", "HD95(mm)"),
            ("iou", "IoU"),
            ("asd_mm", "ASD(mm)"),
        ]
        lines = []
        for key, label in key_map:
            val = metrics.get(key)
            if val is None:
                continue
            if isinstance(val, (int, float)):
                lines.append(f"{label}: {val:.4f}")
            else:
                lines.append(f"{label}: {val}")

        if lines:
            axes[2].text(
                0.03,
                0.03,
                "\n".join(lines),
                transform=axes[2].transAxes,
                fontsize=9,
                color="white",
                bbox=dict(facecolor="black", alpha=0.55, pad=6),
            )

    fig.suptitle(title, y=0.98)
    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.show()


def densify_polyline(poly: np.ndarray, max_step_mm: float = 0.4) -> np.ndarray:
    if max_step_mm <= 0 or poly.shape[0] < 2:
        return poly
    out = [poly[0]]
    for i in range(poly.shape[0] - 1):
        p0 = poly[i]
        p1 = poly[i + 1]
        seg = p1 - p0
        seg_len = float(np.linalg.norm(seg))
        if seg_len <= max_step_mm:
            out.append(p1)
            continue
        n = int(np.ceil(seg_len / max_step_mm))
        for k in range(1, n + 1):
            t = float(k) / float(n)
            out.append(p0 * (1.0 - t) + p1 * t)
    return np.asarray(out, dtype=float)


def read_vtp_geometry(path: Path, shift_world=None, densify_step_mm: float = 0.0):
    must_exist(path)
    reader = vtk.vtkXMLPolyDataReader()
    reader.SetFileName(str(path))
    reader.Update()
    poly = reader.GetOutput()

    pts = poly.GetPoints()
    if pts is None:
        return {"lines": [], "points": np.zeros((0, 3), dtype=float)}

    all_points = np.array([pts.GetPoint(i) for i in range(pts.GetNumberOfPoints())], dtype=float)

    lines = poly.GetLines()
    out_lines = []
    lines.InitTraversal()
    ids = vtk.vtkIdList()
    while lines.GetNextCell(ids):
        if ids.GetNumberOfIds() < 2:
            continue
        arr = np.array([pts.GetPoint(ids.GetId(i)) for i in range(ids.GetNumberOfIds())], dtype=float)
        if densify_step_mm and densify_step_mm > 0:
            arr = densify_polyline(arr, densify_step_mm)
        out_lines.append(arr)

    if shift_world is not None:
        shift = np.asarray(shift_world, dtype=float)
        all_points = all_points + shift[None, :]
        out_lines = [polyline + shift[None, :] for polyline in out_lines]

    return {"lines": out_lines, "points": all_points}


def plot_polylines_3d(polylines, ax, color, lw=1.0, alpha=0.9, max_lines=None, label=None, linestyle='-', mark_endpoints=False):
    if max_lines is not None:
        polylines = polylines[:max_lines]
    first = True
    for poly in polylines:
        ax.plot(
            poly[:, 0], poly[:, 1], poly[:, 2],
            color=color,
            linewidth=lw,
            alpha=alpha,
            linestyle=linestyle,
            label=label if first else None,
        )
        if mark_endpoints and poly.shape[0] >= 2:
            ax.scatter(poly[[0, -1], 0], poly[[0, -1], 1], poly[[0, -1], 2], c=color, s=10, alpha=min(1.0, alpha + 0.1))
        first = False


def set_axes_equal(ax, points: np.ndarray):
    if points.size == 0:
        return
    mins = points.min(axis=0)
    maxs = points.max(axis=0)
    center = (mins + maxs) / 2.0
    radius = float((maxs - mins).max() / 2.0 + 1e-6)
    ax.set_xlim(center[0] - radius, center[0] + radius)
    ax.set_ylim(center[1] - radius, center[1] + radius)
    ax.set_zlim(center[2] - radius, center[2] + radius)


def show_centerline_overlay(
    pred_path: Path,
    gt_path: Path,
    title: str,
    show_pred_panel: bool = True,
    pred_shift_world=None,
    densify_step_mm: float = 0.4,
):
    pred = read_vtp_geometry(pred_path, shift_world=pred_shift_world, densify_step_mm=densify_step_mm)
    gt = read_vtp_geometry(gt_path, shift_world=None, densify_step_mm=densify_step_mm)

    pred_lines = pred["lines"]
    gt_lines = gt["lines"]
    pred_points = pred["points"]
    gt_points = gt["points"]

    print(f"pred: {len(pred_lines)} lines, {pred_points.shape[0]} points")
    print(f"gt:   {len(gt_lines)} lines, {gt_points.shape[0]} points")
    if pred_shift_world is not None:
        print("pred shift (from repair report):", np.asarray(pred_shift_world, dtype=float).tolist())

    all_pts = []
    if gt_points.size > 0:
        all_pts.append(gt_points)
    if pred_points.size > 0:
        all_pts.append(pred_points)
    stacked = np.vstack(all_pts) if all_pts else np.zeros((0, 3), dtype=float)

    if show_pred_panel:
        fig = plt.figure(figsize=(15, 7))
        ax = fig.add_subplot(121, projection="3d")
        ax_pred = fig.add_subplot(122, projection="3d")
    else:
        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection="3d")
        ax_pred = None

    # Overlay panel
    plot_polylines_3d(gt_lines, ax, color="lime", lw=1.0, alpha=0.45, label="GT (lines)", linestyle='--', mark_endpoints=False)
    if pred_lines:
        plot_polylines_3d(pred_lines, ax, color="red", lw=2.3, alpha=0.98, label="Pred (repaired lines)", linestyle='-', mark_endpoints=True)
    elif pred_points.size > 0:
        ax.scatter(pred_points[:, 0], pred_points[:, 1], pred_points[:, 2], s=7, c='red', alpha=0.9, label='Pred (points)')

    if stacked.size > 0:
        set_axes_equal(ax, stacked)

    ax.set_title(title + " [Overlay]")
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.legend(loc="upper right")

    # Pred-only panel
    if ax_pred is not None:
        if pred_lines:
            plot_polylines_3d(pred_lines, ax_pred, color="red", lw=2.6, alpha=1.0, label="Pred only", linestyle='-', mark_endpoints=True)
        elif pred_points.size > 0:
            ax_pred.scatter(pred_points[:, 0], pred_points[:, 1], pred_points[:, 2], s=7, c='red', alpha=0.95, label='Pred (points)')

        if pred_points.size > 0:
            set_axes_equal(ax_pred, pred_points)

        ax_pred.set_title("Pred only (full repaired centerline)")
        ax_pred.set_xlabel("X")
        ax_pred.set_ylabel("Y")
        ax_pred.set_zlabel("Z")
        ax_pred.legend(loc="upper right")

    plt.tight_layout()
    plt.show()


def print_metrics(step_name: str, metrics: dict):
    print(f"[{step_name}] metrics")
    for k, v in metrics.items():
        print(f"  {k}: {v}")



## Step1: segmentation visualization
Green contour = GT mask, Red contour = predicted mask.

In [ ]:
step1 = load_json(OUT_DIR / "step1_segmentation/metrics.json")
print_metrics("step1_segmentation", step1["metrics"])

ct, _, _ = load_nifti(CT_PATH)
gt_mask, _, _ = load_nifti(GT_MASK_PATH)
pred_mask, _, _ = load_nifti(OUT_DIR / "step1_segmentation/totalseg_output/coronary_arteries.nii.gz")

show_seg_overlay(ct, gt_mask > 0.5, pred_mask > 0.5, title=f"{CASE} Step1 (GT vs Pred)", metrics=step1["metrics"])

## Step2: extracted centerline vs GT centerline

In [ ]:
step2 = load_json(OUT_DIR / "step2_centerline/metrics.json")
print_metrics("step2_centerline", step2["metrics"])

pred_cl = OUT_DIR / "step2_centerline/pred_centerline.vtp"
show_centerline_overlay(pred_cl, GT_CL_PATH, title=f"{CASE} Step2: Pred centerline vs GT", show_pred_panel=True)

## Step3: repaired centerline vs baseline and GT

In [ ]:
step3 = load_json(OUT_DIR / "step3_repair/metrics.json")
print_metrics("step3_repair", step3["metrics"])
print("delta:", step3.get("delta", {}))

repaired_cl = OUT_DIR / "step3_repair/repaired.vtp"
repair_report = OUT_DIR / "step3_repair/repair_report.json"
pred_shift = repair_alignment_shift_from_report(repair_report)

show_centerline_overlay(
    repaired_cl,
    GT_CL_PATH,
    title=f"{CASE} Step3: Repaired centerline vs GT",
    show_pred_panel=True,
    pred_shift_world=pred_shift,
    densify_step_mm=0.4,
)



## Step4: feature extraction comparison (上一版口径)
输入：`features_pred` 与 `features_gt`（由分割 mask 提取）。
说明：本版 Step4 不直接读取 Step3 的 repaired centerline。

In [ ]:
pred_summary = load_json(OUT_DIR / "features_pred/summary.json")
gt_summary = load_json(OUT_DIR / "features_gt/summary.json")
step4 = load_json(OUT_DIR / "step4_features/metrics.json")
print_metrics("step4_features", step4["metrics"])
print("[note] Step4 口径: features 来自 segmentation mask，不直接使用 step3 repaired.vtp")
print("pred features:", OUT_DIR / "features_pred")
print("gt features:", OUT_DIR / "features_gt")

pred_lengths = sorted([float(b["length_mm"]) for b in pred_summary.get("branches", [])], reverse=True)
gt_lengths = sorted([float(b["length_mm"]) for b in gt_summary.get("branches", [])], reverse=True)

n = max(len(pred_lengths), len(gt_lengths))
x = np.arange(n)
pred_plot = np.array(pred_lengths + [np.nan] * (n - len(pred_lengths)))
gt_plot = np.array(gt_lengths + [np.nan] * (n - len(gt_lengths)))

plt.figure(figsize=(10, 4))
plt.plot(x, pred_plot, "-o", label="Pred branch length")
plt.plot(x, gt_plot, "-o", label="GT branch length")
plt.xlabel("Branch rank")
plt.ylabel("Length (mm)")
plt.title(f"{CASE} Step4 branch length comparison")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Step5: reconstruction visualization (上一版口径)
输入：Step4 的 `features_pred`，与 GT mask surface 对比。
说明：Step5 不直接读取 Step3 repaired.vtp。

In [ ]:
from skimage import measure
import warnings


def gt_mask_to_mesh(mask_path: Path):
    mask, affine, _ = load_nifti(mask_path)
    binary = (mask > 0.5).astype(np.uint8)
    if binary.sum() == 0:
        raise ValueError(f"Empty GT mask: {mask_path}")
    verts, faces, _, _ = measure.marching_cubes(binary, level=0.5)
    verts_world = voxel_indices_to_world(verts, affine)
    faces_pv = np.hstack([np.full((faces.shape[0], 1), 3, dtype=np.int64), faces.astype(np.int64)]).ravel()
    return verts_world, faces_pv


def mask_boundary_points_world(mask_path: Path, max_points: int = 120000):
    mask, affine, _ = load_nifti(mask_path)
    binary = (mask > 0.5).astype(np.uint8)
    if binary.sum() == 0:
        return np.zeros((0, 3), dtype=float)

    pad = np.pad(binary, 1, mode="constant", constant_values=0)
    center = pad[1:-1, 1:-1, 1:-1]
    neigh_sum = (
        pad[:-2, 1:-1, 1:-1]
        + pad[2:, 1:-1, 1:-1]
        + pad[1:-1, :-2, 1:-1]
        + pad[1:-1, 2:, 1:-1]
        + pad[1:-1, 1:-1, :-2]
        + pad[1:-1, 1:-1, 2:]
    )
    boundary = (center == 1) & (neigh_sum < 6)
    ijk = np.argwhere(boundary)

    if ijk.shape[0] > max_points:
        step = int(np.ceil(ijk.shape[0] / max_points))
        ijk = ijk[::step]

    return voxel_indices_to_world(ijk.astype(float), affine)


def feature_points_world(features_dir: Path, max_points: int = 60000):
    summary_path = features_dir / "summary.json"
    must_exist(summary_path)
    summary = load_json(summary_path)

    chunks = []
    for b in summary.get("branches", []):
        f = features_dir / b["feature_file"]
        if not f.exists():
            continue
        data = np.load(f)
        if "samples_world" in data:
            chunks.append(np.asarray(data["samples_world"], dtype=float))

    if not chunks:
        return np.zeros((0, 3), dtype=float)

    pts = np.vstack(chunks)
    if pts.shape[0] > max_points:
        step = int(np.ceil(pts.shape[0] / max_points))
        pts = pts[::step]
    return pts


def show_step5_fallback(pred_points: np.ndarray, gt_points: np.ndarray, title: str):
    fig = plt.figure(figsize=(15, 7))
    ax1 = fig.add_subplot(121, projection="3d")
    ax2 = fig.add_subplot(122, projection="3d")

    if gt_points.size > 0:
        ax1.scatter(gt_points[:, 0], gt_points[:, 1], gt_points[:, 2], s=0.2, c="lime", alpha=0.18, label="GT boundary")
    if pred_points.size > 0:
        ax1.scatter(pred_points[:, 0], pred_points[:, 1], pred_points[:, 2], s=2.2, c="red", alpha=0.85, label="Pred recon")

    if pred_points.size > 0:
        ax2.scatter(pred_points[:, 0], pred_points[:, 1], pred_points[:, 2], s=2.2, c="red", alpha=0.9, label="Pred recon")

    all_pts = []
    if gt_points.size > 0:
        all_pts.append(gt_points)
    if pred_points.size > 0:
        all_pts.append(pred_points)
    if all_pts:
        set_axes_equal(ax1, np.vstack(all_pts))
    if pred_points.size > 0:
        set_axes_equal(ax2, pred_points)

    ax1.set_title("Overlay: GT boundary (green) + Pred (red)")
    ax2.set_title("Pred only")

    for ax in [ax1, ax2]:
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        ax.legend(loc="upper right")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


step5 = load_json(OUT_DIR / "step5_render/metrics.json")
print_metrics("step5_render", step5["metrics"])
print("[note] Step5 口径: reconstruct from features_pred (from Step4), not direct step3 centerline")

pred_features_dir = OUT_DIR / "features_pred"

use_pyvista = False
try:
    import pyvista as pv
    from vessel_seg.shape import reconstruct_from_features
    use_pyvista = True
except Exception as exc:
    warnings.warn(f"PyVista path unavailable, fallback to matplotlib point-cloud mode. reason: {exc}")

if use_pyvista:
    try:
        try:
            pv.start_xvfb()
        except Exception:
            pass

        pred_mesh = reconstruct_from_features(pred_features_dir, None)
        gt_verts, gt_faces_pv = gt_mask_to_mesh(GT_MASK_PATH)
        gt_mesh = pv.PolyData(gt_verts, gt_faces_pv).clean()

        plotter = pv.Plotter(shape=(1, 2), off_screen=True, window_size=(1600, 700))
        plotter.subplot(0, 0)
        plotter.add_text("Pred reconstructed mesh", font_size=11)
        plotter.add_mesh(pred_mesh, color="tomato", opacity=0.85)
        plotter.view_isometric()

        plotter.subplot(0, 1)
        plotter.add_text("GT mask surface", font_size=11)
        plotter.add_mesh(gt_mesh, color="seagreen", opacity=0.85)
        plotter.view_isometric()

        img = plotter.screenshot(return_img=True)
        plotter.close()

        plt.figure(figsize=(14, 6))
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"{CASE} Step5 mesh render (left=Pred, right=GT)")
        plt.tight_layout()
        plt.show()

    except Exception as exc:
        warnings.warn(f"PyVista render failed, fallback to matplotlib point-cloud mode. reason: {exc}")
        pred_pts = feature_points_world(pred_features_dir)
        gt_pts = mask_boundary_points_world(GT_MASK_PATH)
        print(f"fallback points: pred={pred_pts.shape[0]}, gt={gt_pts.shape[0]}")
        show_step5_fallback(pred_pts, gt_pts, title=f"{CASE} Step5 fallback visualization")
else:
    pred_pts = feature_points_world(pred_features_dir)
    gt_pts = mask_boundary_points_world(GT_MASK_PATH)
    print(f"fallback points: pred={pred_pts.shape[0]}, gt={gt_pts.shape[0]}")
    show_step5_fallback(pred_pts, gt_pts, title=f"{CASE} Step5 fallback visualization")



## 三图总览：Pred 分割 / 修复+特征 / GT 分割
左图：CT + TotalSegmentator 预测分割（红）  
中图：修复后中心线（橙）+ 提取的特征采样点（青）  
右图：CT + GT 分割（绿）  
注：该中图用于可视化展示；上一版指标口径下，Step4/Step5 的计算输入以分割特征链路为准。

In [ ]:
# --- 3-panel quick comparison (no pyvista required) ---
from matplotlib.lines import Line2D


def load_feature_samples(features_dir: Path, max_points: int = 15000) -> np.ndarray:
    summary = load_json(features_dir / "summary.json")
    pts = []
    for b in summary.get("branches", []):
        f = features_dir / b.get("feature_file", "")
        if not f.exists():
            continue
        data = np.load(f)
        if "samples_world" in data:
            pts.append(np.asarray(data["samples_world"], dtype=float))
    if not pts:
        return np.zeros((0, 3), dtype=float)
    out = np.vstack(pts)
    if out.shape[0] > max_points:
        step = int(np.ceil(out.shape[0] / max_points))
        out = out[::step]
    return out


def show_three_panel_overview(case_name: str):
    ct, _, _ = load_nifti(CT_PATH)
    pred_mask, _, _ = load_nifti(OUT_DIR / "step1_segmentation/totalseg_output/coronary_arteries.nii.gz")
    gt_mask, _, _ = load_nifti(GT_MASK_PATH)

    center = mask_center((gt_mask > 0.5) | (pred_mask > 0.5))
    z, y, x = center

    repair_report = OUT_DIR / "step3_repair/repair_report.json"
    pred_shift = repair_alignment_shift_from_report(repair_report)
    repaired_geo = read_vtp_geometry(OUT_DIR / "step3_repair/repaired.vtp", shift_world=pred_shift, densify_step_mm=0.4)
    repaired_lines = repaired_geo["lines"]
    repaired_points = repaired_geo["points"]
    feat_points = load_feature_samples(OUT_DIR / "features_pred")

    step1 = load_json(OUT_DIR / "step1_segmentation/metrics.json")
    step3 = load_json(OUT_DIR / "step3_repair/metrics.json")

    fig = plt.figure(figsize=(20, 6))
    ax1 = fig.add_subplot(131)
    ax2 = fig.add_subplot(132, projection="3d")
    ax3 = fig.add_subplot(133)

    # panel 1: CT + pred mask
    ax1.imshow(ct[z, :, :].T, cmap="gray", origin="lower")
    ax1.contour((pred_mask[z, :, :] > 0.5).T, levels=[0.5], colors=["red"], linewidths=1.4)
    ax1.set_title(f"{case_name} | CT + TotalSeg Pred (z={z})")
    ax1.set_axis_off()
    p1_legend = [
        Line2D([0], [0], color="gray", lw=4, label="CT"),
        Line2D([0], [0], color="red", lw=2, label="TotalSeg Pred"),
    ]
    ax1.legend(handles=p1_legend, loc="lower right", frameon=True)

    # panel 2: repaired centerline + features
    if repaired_lines:
        plot_polylines_3d(
            repaired_lines,
            ax2,
            color="orange",
            lw=2.0,
            alpha=0.95,
            label="Repaired centerline",
            mark_endpoints=True,
        )
    elif repaired_points.size > 0:
        ax2.scatter(repaired_points[:, 0], repaired_points[:, 1], repaired_points[:, 2], s=6, c="orange", alpha=0.9, label="Repaired centerline pts")

    if feat_points.size > 0:
        ax2.scatter(feat_points[:, 0], feat_points[:, 1], feat_points[:, 2], s=3, c="deepskyblue", alpha=0.55, label="Feature samples")

    all_pts = []
    if repaired_points.size > 0:
        all_pts.append(repaired_points)
    if feat_points.size > 0:
        all_pts.append(feat_points)
    if all_pts:
        set_axes_equal(ax2, np.vstack(all_pts))

    ax2.set_title(f"{case_name} | Repaired + Feature extraction")
    ax2.set_xlabel("X")
    ax2.set_ylabel("Y")
    ax2.set_zlabel("Z")
    ax2.legend(loc="upper right")

    # panel 3: CT + gt mask
    ax3.imshow(ct[z, :, :].T, cmap="gray", origin="lower")
    ax3.contour((gt_mask[z, :, :] > 0.5).T, levels=[0.5], colors=["lime"], linewidths=1.4)
    ax3.set_title(f"{case_name} | CT + GT Mask (z={z})")
    ax3.set_axis_off()
    p3_legend = [
        Line2D([0], [0], color="gray", lw=4, label="CT"),
        Line2D([0], [0], color="lime", lw=2, label="GT mask"),
    ]
    ax3.legend(handles=p3_legend, loc="lower right", frameon=True)

    # text metrics on figure bottom
    s1 = step1.get("metrics", {})
    s3 = step3.get("metrics", {})
    txt = (
        f"Step1 Dice={s1.get('dice', float('nan')):.4f}, HD95={s1.get('hd95_mm', float('nan')):.3f} mm | "
        f"Step3 pred2gt_mean={s3.get('pred2gt_mean', float('nan')):.3f} mm, "
        f"coverage_pred@1mm={s3.get('coverage_pred@1mm', float('nan')):.3f}"
    )
    fig.text(0.5, 0.01, txt, ha="center", va="bottom", fontsize=10)

    plt.tight_layout(rect=[0, 0.05, 1, 1])
    plt.show()


show_three_panel_overview(CASE)



## Metrics summary table

In [ ]:
steps = [
    "step1_segmentation",
    "step2_centerline",
    "step3_repair",
    "step4_features",
    "step5_render",
]
rows = []
for s in steps:
    data = load_json(OUT_DIR / s / "metrics.json")
    row = {"step": s}
    row.update(data.get("metrics", {}))
    rows.append(row)

if pd is not None:
    display(pd.DataFrame(rows))
else:
    for row in rows:
        print(row)